In [ ]:
import os, kagglehub, torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 데이터 로드 및 분할
path = kagglehub.dataset_download("mostafaabla/garbage-classification")
data_dir = next(r for r, d, _ in os.walk(path) if len(d) == 12)

dataset = datasets.ImageFolder(data_dir, transform=transforms.Compose([
    transforms.Resize((128, 128)), transforms.ToTensor()
]))
X = torch.stack([dataset[i][0] for i in range(len(dataset))])
Y = torch.tensor(dataset.targets)

idx = torch.randperm(len(dataset)).tolist()
n_tr, n_va = int(len(idx) * 0.7), int(len(idx) * 0.15)
tr_idx, va_idx, te_idx = idx[:n_tr], idx[n_tr:n_tr + n_va], idx[n_tr + n_va:]

class DS(Dataset):
    def __init__(self, ids, tf=None): self.ids, self.tf = ids, tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        x = X[self.ids[i]]
        return (self.tf(x) if self.tf else x), Y[self.ids[i]]

tf_aug = transforms.Compose([
    transforms.RandomRotation(30),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
])
train_loader = DataLoader(DS(tr_idx, tf_aug), batch_size=64, shuffle=True)
val_loader   = DataLoader(DS(va_idx), batch_size=256)
test_loader  = DataLoader(DS(te_idx), batch_size=256)

# 2. 최종 모델 정의 (CNN + BatchNorm)
def conv_block(i, o):
    return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2)]

torch.manual_seed(42)
model = nn.Sequential(
    *conv_block(3, 32), *conv_block(32, 64), *conv_block(64, 64),
    nn.Flatten(), nn.Linear(64 * 16 * 16, 256), nn.BatchNorm1d(256), nn.ReLU(),
    nn.Linear(256, len(dataset.classes))
).to(device)

# 3. 학습 및 평가
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-4)

def evaluate(loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total

for epoch in range(1, 31):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    if epoch % 5 == 0 or epoch == 1:
        va_loss, va_acc = evaluate(val_loader)
        print(f"Epoch {epoch:2d} | Val Loss: {va_loss:.4f}, Val Acc: {va_acc:.4f}")

te_loss, te_acc = evaluate(test_loader)
print(f"\n최종 테스트 Loss: {te_loss:.4f} | 정확도: {te_acc:.4f}")